# Задание 3. Генерация коммерческого описания снимка

Компания «Экзосфера» хочет быстрее готовить описания снимков. По готовой текстовой сводке обучите LSTM предсказывать следующий токен и сравните продолжения с готовым трансформером. Результат проверяет специалист.

## Аудит готовых выборок и проверка поля text

Опишите роль системы компьютерного зрения, языковой модели и человека. Загрузите обучающие и валидационные тексты и `commercial_descriptions_eval_ids.csv`. Все 450 валидационных текстов нужны для расчёта ошибки при обучении, выбранные 100 — для оценки генерации на CPU. Файл содержит только `text_id`, найдите соответствующие строки в валидационном файле. Проверьте структуру, пропуски, повторы, длины, примеры и шаблон `text`. Сохраните выданное разделение.

## Токенизация, словарь, PAD, UNK и EOS

Приведите текст к нижнему регистру и выполните пословную токенизацию, сохраняя слова, числа и знаки пунктуации отдельными токенами. Постройте словарь только по обучающим текстам, примените `min_freq` и измерьте долю неизвестных токенов на валидационной выборке. Добавьте PAD, UNK и EOS. Отделите токенизацию корпуса от подготовки продолжений к оценке ROUGE.

## TextDataset и отдельные DataLoader

Используйте TextDataset с неперекрывающимися отрезками длиной `seq_len=35`: начало отрезка `i * seq_len`, вход — 35 токенов, целевая последовательность — те же позиции со сдвигом на один токен. Модель предсказывает следующий токен для каждой позиции. Сохраните все целевые позиции: округляйте число отрезков вверх, короткий последний отрезок дополните PAD, исключите PAD из функции потерь и задайте `drop_last=False` в обоих загрузчиках. Для последовательности из 73 токенов нужно сохранить 72 целевые позиции: два полных отрезка покрывают 70, третий сохраняет оставшиеся две. Соберите отдельные обучающий и валидационный корпуса с EOS между текстами. Полные требования приведены в уроке проекта.

## Архитектура Embedding → LSTM → Linear

Реализуйте изученную архитектуру. Выберите 1–2 слоя, размер эмбеддинга 64–128, скрытого состояния 128–256, пакет 32–64, скорость обучения 0.0005–0.002 и 3–10 эпох.

## Обучение и кривые ошибки

Обучите модель на CPU, покажите ошибку на обучающей и валидационной выборках и интерпретируйте динамику.

## Сохранение, загрузка и генерация

Сохраните `state_dict`, повторно загрузите модель и реализуйте жадную генерацию и выборку с температурой. Веса храните локально вне Git.

## Фиксированные 100 префиксов и оценка LSTM

Получите продолжения LSTM жадной генерацией на 100 выданных идентификаторах: до 20 предсказаний токена с остановкой при EOS. После общей подготовки оставьте до 20 слов по протоколу и измерьте время генерации. Покажите число различных подготовленных продолжений, долю самого частого и пустые результаты. Авторский ориентир 24 варианта зависит от конфигурации, словаря, эпох и случайного зерна; это диагностика, а не порог сдачи. Объясните повторения и ограничения.

## Предоставленный трансформер

Загрузите `ai-forever/rugpt3small_based_on_gpt2` в зафиксированной курсом ревизии `a9307e696cd3c5b7f953ff4cb19d76a4d81821d5`. Пример загрузки приведён в протоколе проекта. Включите `eval()`, используйте CPU и `torch.no_grad()` без дообучения модели. Для каждого из 100 префиксов выполните жадную генерацию с `do_sample=False` и `max_new_tokens=64`, затем декодируйте только новые токены.

## Подготовка продолжений и расчёт ROUGE

Реализуйте подготовку текстов и расчёт ROUGE по протоколу в уроке проекта: нижний регистр, словесные токены Unicode, счётчики n-грамм, пересечение с учётом повторов и F1 по точности и полноте совпадений. Одинаково обработайте продолжения обеих моделей и эталон, оставьте первые 20 словесных токенов каждого текста. На одинаковых русскоязычных строках из нескольких слов получите ROUGE-1 F1 и ROUGE-2 F1, равные 1,0; при пустом пересечении или отсутствии n-грамм верните 0. Рассчитайте обе метрики для каждой пары и отдельно усредните по всем 100 примерам, включая пустые продолжения. Реализацию напишите непосредственно в ноутбуке.

## Таблица фактической согласованности

Для 5–10 пар заполните таблицу: факты CV, поддержанные утверждения, противоречия, неподтверждённые утверждения, итог. Отдельно отмечайте неверно скопированные площадь, дату, разрешение и облачность.

## Рекомендация и ограничения

Сравните качество, длину, время и ресурсы: CPU, число параметров, необходимость загрузки и кэша трансформера. Выберите архитектуру и зафиксируйте обязательную проверку человеком.